# Experiment: MediaPipe Export and Features

What this notebook teaches:
- Run frame-by-frame MediaPipe inference on a short clip.
- Export canonical keypoints to CSV/JSON for downstream analysis.
- Compute joint angles, smoothed curves, and angular velocities.
- Save compact benchmark and feature artifacts.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumeyye-agac/human-pose-estimation-experiments.git"
REPO_DIR = Path("/content/human-pose-estimation-experiments")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    raise RuntimeError("Run this notebook from the repository root.")

if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Using repo root: {repo_root}")


In [ ]:
def pip_install(*packages: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

pip_install("mediapipe", "opencv-python-headless", "matplotlib", "numpy", "pandas", "scipy")


In [ ]:
import json
import urllib.request
import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import pandas as pd
from posebench.benchmark import BenchmarkConfig, benchmark_backend, write_json
from posebench.export import export_frames_to_csv, export_frames_to_json
from posebench.features import compute_angular_velocity, extract_joint_angles, smooth_series, summarize_series_features
from posebench.keypoints_schema import map_tool_keypoints_to_canonical
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle.mp4"
video_path = repo_root / "assets" / "sample_input_motion.mp4"
if not video_path.exists():
    urllib.request.urlretrieve(video_url, video_path)
cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError("Could not open sample video")
pose = mp.solutions.pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)
frames = []
frame_index = 0
max_frames = 48
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
while frame_index < max_frames:
    ok, bgr = cap.read()
    if not ok:
        break
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    result = pose.process(rgb)
    h, w = bgr.shape[:2]
    if result.pose_landmarks:
        points = [
            {"x": lm.x * w, "y": lm.y * h, "confidence": lm.visibility}
            for lm in result.pose_landmarks.landmark
        ]
    else:
        points = []
    canonical = map_tool_keypoints_to_canonical("mediapipe", points, min_confidence=0.1)
    frames.append(
        {
            "frame_index": frame_index,
            "timestamp_ms": (frame_index / fps) * 1000.0,
            "person_id": 0,
            "tool": "mediapipe",
            "schema": "coco17",
            "keypoints": canonical,
        }
    )
    frame_index += 1
cap.release()
csv_path = repo_root / "results" / "mediapipe_sequence_canonical.csv"
json_path = repo_root / "results" / "mediapipe_sequence_canonical.json"
export_frames_to_csv(frames, csv_path)
export_frames_to_json(frames, json_path)
joint_triplets = {
    "left_knee_angle": ("left_hip", "left_knee", "left_ankle"),
    "right_elbow_angle": ("right_shoulder", "right_elbow", "right_wrist"),
    "left_hip_angle": ("left_shoulder", "left_hip", "left_knee"),
}
angles_df = extract_joint_angles(frames, joint_triplets, min_confidence=0.2)
angles_df["left_knee_angle_smooth"] = smooth_series(angles_df["left_knee_angle"], method="savgol")
angles_df["left_knee_velocity"] = compute_angular_velocity(angles_df["left_knee_angle_smooth"], fps=fps)
summary = {
    column: summarize_series_features(angles_df[column].to_numpy())
    for column in ["left_knee_angle", "right_elbow_angle", "left_hip_angle"]
}
summary_path = repo_root / "results" / "mediapipe_features_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
plt.figure(figsize=(9, 4))
for column in ["left_knee_angle", "right_elbow_angle", "left_hip_angle"]:
    plt.plot(angles_df["frame_index"], angles_df[column], label=column)
plt.xlabel("Frame")
plt.ylabel("Angle (deg)")
plt.title("Joint angle time series")
plt.legend()
plt.show()
class MPBackend:
    name = "mediapipe"
    def __init__(self, pose_model):
        self.pose_model = pose_model
    def infer(self, frame: np.ndarray):
        return self.pose_model.process(frame)
bench_frame = np.random.default_rng(17).integers(0, 255, size=(480, 640, 3), dtype=np.uint8)
bench_result = benchmark_backend(
    backend=MPBackend(pose),
    frames=[bench_frame],
    config=BenchmarkConfig(warmup_frames=8, measured_frames=40, repeat=1),
)
write_json(bench_result, repo_root / "results" / "benchmark_raw_mediapipe_sequence.json")
print("Saved:")
print("-", csv_path)
print("-", json_path)
print("-", summary_path)
